In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


/bin/bash: line 1: nvidia-smi: command not found


In [3]:
!pip install networkx
import networkx as nx

In [4]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [5]:
community_dict_stimOff = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/community_dict.pt', weights_only=False)

In [6]:
community_dict_stimOn = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/StimOn/community_stimOn_dict.pt', weights_only=False)

In [15]:
################################################################################
import subprocess
import sys
required = {'ONE-api', 'brain', 'ibllib'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *required])

from one.api import ONE
from brainbox.io.one import SessionLoader, SpikeSortingLoader

from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
br = ba.regions
ba.compute_regions_volume()

Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/average_template_25.nrrd Bytes: 32998960


100%|██████████| 31.470260620117188/31.470260620117188 [00:00<00:00, 139.44it/s]


Downloading: /root/Downloads/ONE/openalyx.internationalbrainlab.org/histology/ATLAS/Needles/Allen/annotation_25.nrrd Bytes: 4035363


100%|██████████| 3.848422050476074/3.848422050476074 [00:00<00:00, 39.83it/s]


In [18]:
len(br.acronym)

2655

In [19]:
len(br.name)

2655

In [30]:
resolution = 1.0
br_dict = {}
br_list = []
br_name = []
Cosmos_name = []
community_stimOff_list = []
community_stimOn_list = []
Cosmos_list = []
for acronym in acronym_list:
    br_list.append(acronym)
    Cosmos_acronym = br.acronym2acronym(acronym, mapping='Cosmos')[0]
    Cosmos_list.append(Cosmos_acronym)
    br_name.append(br.name[np.argwhere(br.acronym == acronym)[0]].item())
    Cosmos_name.append(br.name[np.argwhere(br.acronym == Cosmos_acronym)[0]].item())
    if acronym in community_dict_stimOff[resolution]['communities_acronym']:
        community_stimOff_list.append(community_dict_stimOff[resolution]['communities_label'][np.argwhere(community_dict_stimOff[resolution]['communities_acronym'] == acronym).flatten()].item())
    else:
        community_stimOff_list.append(-1)
    if acronym in community_dict_stimOn[resolution]['communities_acronym']:
        community_stimOn_list.append(community_dict_stimOn[resolution]['communities_label'][np.argwhere(community_dict_stimOn[resolution]['communities_acronym'] == acronym).flatten()].item())
    else:
        community_stimOn_list.append(-1)


br_dict = pd.DataFrame(
    {
        'Allen acronym':br_list,
        'Allen name':br_name,
        'Cosmos acronym':Cosmos_list,
        'Cosmos name':Cosmos_name,
        'Inter-trial state':community_stimOff_list,
        'Audio-visual cue':community_stimOn_list,

    }
)

In [31]:
br_dict.to_csv('br_dict.csv', index=False)

In [26]:
br.name[np.argwhere(br.acronym == 'CA1')[0]].item()

'Field CA1'

In [21]:
Cosmos_list

['Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isocortex',
 'Isoc